In [ ]:
import numpy as np
from typing import List, Tuple, Dict, Optional

DropList = List[Tuple[Tuple[int, int], float]]

def _gini(x: np.ndarray) -> float:
    """Gini coefficient of a 1‑D array"""
    if x.size == 0:
        return 0.0
    if np.amin(x) < 0:
        x = x - np.amin(x)
    x = np.sort(x)
    n   = x.size
    cum = np.cumsum(x, dtype=float)
    return (n + 1 - 2 * np.sum(cum) / cum[-1]) / n

def _summarise(drops: DropList, worst_percent: float = 5.0) -> Dict[str, float]:
    """Return summary stats; all zeros if `drops` is empty."""
    if len(drops) == 0:
        return {k: 0.0 for k in ["mean", "median", "worst_k_mean", "single_worst", "gini"]}

    arr = np.asarray([d for _, d in drops], dtype=float)
    k   = max(1, int(np.ceil(len(arr) * worst_percent / 100)))
    worst_k_mean = arr[np.argsort(arr)[-k:]].mean()

    return {
        "mean"        : float(arr.mean()),
        "median"      : float(np.median(arr)),
        "worst_k_mean": float(worst_k_mean),
        "single_worst": float(arr.max()),
        "gini"        : float(_gini(arr)),
    }

def compare_teacher_student(
    teacher_heads : DropList,
    student_heads : DropList,
    teacher_mlp   : Optional[DropList] = None,
    student_mlp   : Optional[DropList] = None,
    teacher_params: int = 1,
    student_params: int = 1,
    worst_percent : float = 5.0,
) -> Dict[str, Dict[str, float]]:
    """
    Return compression factor, per‑family stats, and slope.
    """
    teacher_mlp   = teacher_mlp   or []
    student_mlp   = student_mlp   or []

    C = 1.0 - student_params / teacher_params

    head_stats_t = _summarise(teacher_heads , worst_percent)
    head_stats_s = _summarise(student_heads , worst_percent)
    mlp_stats_t  = _summarise(teacher_mlp   , worst_percent)
    mlp_stats_s  = _summarise(student_mlp   , worst_percent)

    provided_types = 1 + (len(teacher_mlp) > 0 or len(student_mlp) > 0)
    mean_drop_t = (head_stats_t["mean"] + mlp_stats_t["mean"]) / provided_types
    mean_drop_s = (head_stats_s["mean"] + mlp_stats_s["mean"]) / provided_types

    slope = (mean_drop_s - mean_drop_t) / C if C != 0 else 0.0

    return {
        "compression_C": C,
        "teacher": {
            "heads": head_stats_t,
            "mlp"  : mlp_stats_t,
            "mean_all": mean_drop_t,
        },
        "student": {
            "heads": head_stats_s,
            "mlp"  : mlp_stats_s,
            "mean_all": mean_drop_s,
        },
        "robustness_slope_pp_per_C": slope,
    }


## Numeral seq completion:

In [2]:

texts = ['Van done in 1. Hat done in 2. Ring done in 3. Desk done in 4. Sun done in', 'Ice done in 2. Snow done in 3. Watch done in 4. Sun done in 5. Table done in', 'Ring done in 3. Moon done in 4. Queen done in 5. Book done in 6. Rose done in', 'Queen done in 4. Oil done in 5. Rose done in 6. Desk done in 7. Car done in', 'Light done in 5. Arm done in 6. Road done in 7. Book done in 8. Ice done in', 'Ball done in 6. Cow done in 7. Book done in 8. Rose done in 9. Key done in', 'Road done in 7. Key done in 8. Ocean done in 9. Key done in 10. Queen done in', 'House done in 8. Rose done in 9. Key done in 10. Hat done in 11. Van done in', 'Ring done in 1. Car done in 2. Apple done in 3. Pear done in 4. Moon done in', 'Star done in 2. Sun done in 3. Road done in 4. Queen done in 5. Box done in', 'Hill done in 3. Ant done in 4. Apple done in 5. House done in 6. Hat done in', 'Chair done in 4. Van done in 5. Orange done in 6. Queen done in 7. Zip done in', 'Gate done in 5. Desk done in 6. Wolf done in 7. Rain done in 8. Flag done in', 'Queen done in 6. House done in 7. Light done in 8. Cat done in 9. Moon done in', 'Zip done in 7. Car done in 8. Ring done in 9. Hand done in 10. Fish done in', 'Snake done in 8. Queen done in 9. Window done in 10. Ear done in 11. Orange done in', 'Ice done in 1. Sand done in 2. Desk done in 3. Van done in 4. Hat done in', 'Camera done in 2. Watch done in 3. Dog done in 4. Book done in 5. Desk done in', 'Rain done in 3. Dog done in 4. Rat done in 5. Nut done in 6. Ocean done in', 'Night done in 4. Hat done in 5. Wall done in 6. Book done in 7. Tree done in', 'Gate done in 5. Chair done in 6. Ring done in 7. Hill done in 8. Car done in', 'Ax done in 6. Fan done in 7. Cat done in 8. Ring done in 9. Ring done in', 'Rose done in 7. Night done in 8. Van done in 9. Orange done in 10. Apple done in', 'Orange done in 8. Ear done in 9. Book done in 10. Ring done in 11. Bird done in', 'Ball done in 1. Wind done in 2. Wallet done in 3. Tree done in 4. Sand done in', 'Table done in 2. Jam done in 3. Apple done in 4. Pear done in 5. Ice done in', 'Gate done in 3. Train done in 4. Hat done in 5. Fan done in 6. Key done in', 'Window done in 4. Desk done in 5. Year done in 6. Van done in 7. Camera done in', 'Car done in 5. Watch done in 6. Oil done in 7. Queen done in 8. Fan done in', 'Mouse done in 6. Rose done in 7. Table done in 8. Clock done in 9. Queen done in', 'Year done in 7. Wheel done in 8. Wolf done in 9. Arm done in 10. Ax done in', 'Key done in 8. Ring done in 9. Night done in 10. Fan done in 11. Clock done in', 'Glass done in 1. Key done in 2. Nut done in 3. Rat done in 4. Van done in', 'Star done in 2. Pear done in 3. Bug done in 4. Bird done in 5. Ring done in', 'Road done in 3. Wind done in 4. Rat done in 5. Wolf done in 6. Map done in', 'Car done in 4. Ant done in 5. Rose done in 6. Van done in 7. Wolf done in', 'Zip done in 5. Hat done in 6. Wind done in 7. Rain done in 8. Car done in', 'Ring done in 6. Light done in 7. Jar done in 8. Dog done in 9. Fish done in', 'Bug done in 7. Key done in 8. Gate done in 9. Orange done in 10. Ring done in', 'Flag done in 8. Fire done in 9. Pear done in 10. Ax done in 11. Ear done in', 'Ring done in 1. Flag done in 2. Queen done in 3. Map done in 4. Camera done in', 'Desk done in 2. Fire done in 3. Desk done in 4. Road done in 5. Watch done in', 'Queen done in 3. House done in 4. Flag done in 5. Tree done in 6. Ring done in', 'Desk done in 4. Window done in 5. Fish done in 6. Jar done in 7. Hat done in', 'Snake done in 5. Key done in 6. Glass done in 7. Van done in 8. House done in', 'Camera done in 6. Year done in 7. Rose done in 8. Map done in 9. Fire done in', 'Rose done in 7. Snake done in 8. Corn done in 9. Desk done in 10. Zip done in', 'Table done in 8. Key done in 9. Eye done in 10. Van done in 11. Key done in', 'Star done in 1. Hat done in 2. Star done in 3. Eye done in 4. Gold done in', 'Gate done in 2. Queen done in 3. Camera done in 4. Zip done in 5. Ocean done in', 'Fan done in 3. Corn done in 4. Apple done in 5. Van done in 6. Cat done in', 'Key done in 4. Gold done in 5. Star done in 6. Queen done in 7. Arm done in', 'Van done in 5. Snake done in 6. Car done in 7. Star done in 8. Watch done in', 'Jar done in 6. Flag done in 7. Window done in 8. Sun done in 9. Van done in', 'Wolf done in 7. Watch done in 8. Ring done in 9. Desk done in 10. Car done in', 'Cow done in 8. Tree done in 9. Book done in 10. Wall done in 11. Flag done in', 'Van done in 1. Wall done in 2. Ring done in 3. Car done in 4. Key done in', 'Jar done in 2. Wallet done in 3. Fish done in 4. Watch done in 5. Orange done in', 'Watch done in 3. Orange done in 4. Table done in 5. Corn done in 6. Van done in', 'Lake done in 4. Star done in 5. Fan done in 6. Fan done in 7. Night done in', 'Fan done in 5. Ball done in 6. Desk done in 7. Jam done in 8. Tree done in', 'Key done in 6. Jar done in 7. Night done in 8. Fish done in 9. Fan done in', 'Ice done in 7. Rose done in 8. Wall done in 9. Road done in 10. Window done in', 'Tree done in 8. Hill done in 9. Wall done in 10. Queen done in 11. Desk done in', 'Gold done in 1. King done in 2. Queen done in 3. Desk done in 4. Night done in', 'Van done in 2. Sand done in 3. Tree done in 4. Car done in 5. Cow done in', 'Pear done in 3. Eye done in 4. Lake done in 5. Fan done in 6. Light done in', 'Jam done in 4. Road done in 5. Corn done in 6. Ice done in 7. Pen done in', 'Iron done in 5. Ring done in 6. Star done in 7. Van done in 8. Star done in', 'Bird done in 6. Wheel done in 7. Cat done in 8. Desk done in 9. Desk done in', 'Fish done in 7. Gold done in 8. King done in 9. Ax done in 10. Jam done in', 'Desk done in 8. House done in 9. Cow done in 10. Hill done in 11. Gate done in', 'Nut done in 1. Light done in 2. Sand done in 3. Wall done in 4. Key done in', 'Tree done in 2. Fan done in 3. Car done in 4. Desk done in 5. Ring done in', 'Orange done in 3. Ocean done in 4. Sun done in 5. Bug done in 6. Queen done in', 'Orange done in 4. House done in 5. Watch done in 6. Corn done in 7. Ring done in', 'Wind done in 5. Orange done in 6. Rose done in 7. Zip done in 8. Queen done in', 'Iron done in 6. Cat done in 7. Glass done in 8. Box done in 9. Rose done in', 'Clock done in 7. Zip done in 8. Cat done in 9. Snow done in 10. Ant done in', 'Rose done in 8. Ear done in 9. Rose done in 10. Rat done in 11. Orange done in', 'Road done in 1. Ring done in 2. Pen done in 3. Oil done in 4. Camera done in', 'House done in 2. Zip done in 3. Queen done in 4. Snow done in 5. Lake done in', 'Van done in 3. Tree done in 4. Wall done in 5. Rat done in 6. King done in', 'Ball done in 4. Hill done in 5. Clock done in 6. Ear done in 7. Van done in', 'Night done in 5. Ice done in 6. Apple done in 7. Tree done in 8. House done in', 'Clock done in 6. Orange done in 7. Queen done in 8. Desk done in 9. Arm done in', 'Map done in 7. Sun done in 8. Clock done in 9. Cat done in 10. Ball done in', 'Desk done in 8. Key done in 9. Star done in 10. Cow done in 11. Rose done in', 'Ring done in 1. Fan done in 2. Window done in 3. Tree done in 4. Chair done in', 'House done in 2. Nut done in 3. Queen done in 4. Zip done in 5. Rose done in', 'Window done in 3. Gate done in 4. Zip done in 5. Key done in 6. Snow done in', 'Corn done in 4. Ice done in 5. Eye done in 6. Light done in 7. Desk done in', 'Hill done in 5. Snake done in 6. Orange done in 7. Road done in 8. Flag done in', 'Queen done in 6. Window done in 7. Zip done in 8. Cow done in 9. Fire done in', 'Star done in 7. Hat done in 8. Chair done in 9. Car done in 10. Ring done in', 'Desk done in 8. Watch done in 9. Desk done in 10. Ax done in 11. Queen done in', 'Train done in 1. Snow done in 2. Cow done in 3. Ring done in 4. Queen done in', 'Window done in 2. Pen done in 3. Ax done in 4. Iron done in 5. Nut done in', 'Desk done in 3. Flag done in 4. Fan done in 5. Hat done in 6. Chair done in', 'King done in 4. Star done in 5. Cow done in 6. Wallet done in 7. Fan done in', 'Star done in 5. Watch done in 6. Car done in 7. Van done in 8. Queen done in']

import torch
import numpy as np
import re
from transformers import AutoTokenizer, BertForMaskedLM, DistilBertForMaskedLM

device = "cuda" if torch.cuda.is_available() else "cpu"

bert_model = BertForMaskedLM.from_pretrained("bert-base-uncased").to(device)
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
distilbert_model = DistilBertForMaskedLM.from_pretrained("distilbert-base-uncased").to(device)
distilbert_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

bert_model.eval()
distilbert_model.eval()

def normalize_numerals(texts):
    normalized_texts = []
    for text in texts:
        matches = list(re.finditer(r'\b\d+\b', text))
        mapping = {match.group(): str(i + 1) for i, match in enumerate(matches)}
        normalized_text = re.sub(r'\b\d+\b', lambda m: mapping[m.group()], text)
        normalized_texts.append(normalized_text)
    return normalized_texts

texts = [t + " [MASK]" for t in normalize_numerals(texts)]

def compute_logit_diff(model, tokenizer, texts, target_token, distractor_token):
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True).to(device)
    mask_token_index = (inputs.input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)
    with torch.no_grad():
        logits = model(**inputs).logits
    diffs = []
    for b, pos in zip(*mask_token_index):
        logits_slice = logits[b, pos]
        target_logit = logits_slice[tokenizer.convert_tokens_to_ids(target_token)]
        distractor_logit = logits_slice[tokenizer.convert_tokens_to_ids(distractor_token)]
        diffs.append((target_logit - distractor_logit).item())
    return np.mean(diffs)

target_token = '5'
distractor_token = '4'
bert_diff = compute_logit_diff(bert_model, bert_tokenizer, texts, target_token, distractor_token)
distilbert_diff = compute_logit_diff(distilbert_model, distilbert_tokenizer, texts, target_token, distractor_token)

print(f"BERT logit difference: {bert_diff:.4f}")
print(f"DistilBERT logit difference: {distilbert_diff:.4f}")

import torch
import numpy as np
import re
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict
from torch.nn.functional import cosine_similarity
from transformers import (AutoTokenizer, BertForMaskedLM, DistilBertForMaskedLM)

device = "cuda" if torch.cuda.is_available() else "cpu"
teacher_str = "bert-base-uncased"
student_str = "distilbert-base-uncased"

teacher = BertForMaskedLM.from_pretrained(teacher_str).to(device)
student = DistilBertForMaskedLM.from_pretrained(student_str).to(device)
tokenizer_t = AutoTokenizer.from_pretrained(teacher_str)
tokenizer_s = AutoTokenizer.from_pretrained(student_str)

teacher.eval()
student.eval()

def normalize_numerals(texts):
    normalized_texts = []
    for text in texts:
        matches = list(re.finditer(r'\b\d+\b', text))
        mapping = {match.group(): str(i + 1) for i, match in enumerate(matches)}
        normalized_text = re.sub(r'\b\d+\b', lambda m: mapping[m.group()], text)
        normalized_texts.append(normalized_text)
    return normalized_texts

texts = [t + " [MASK]" for t in normalize_numerals(texts)]

def get_logits(model, tokenizer, texts):
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True).to(device)
    mask_token_index = (inputs.input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)
    with torch.no_grad():
        outputs = model(**inputs, return_dict=True)
    return outputs.logits, inputs, mask_token_index

def compute_logit_diff(logits, input_ids, mask_token_index, tokenizer):
    diffs = []
    for b, pos in zip(*mask_token_index):
        logits_slice = logits[b, pos]
        correct_logit = logits_slice[tokenizer.convert_tokens_to_ids(target_token)]
        incorrect_logit = logits_slice[tokenizer.convert_tokens_to_ids(distractor_token)]
        diffs.append((correct_logit - incorrect_logit).item())
    return np.mean(diffs)

def get_ablation_hook(head_idx):
    def ablate_head(module, input, output):
        if isinstance(output, tuple):
            attn_output = output[0]
            attn_output[:, head_idx] = 0
            return (attn_output,) + output[1:]
        output[:, head_idx] = 0
        return output
    return ablate_head

def get_mlp_ablation_hook():
    return lambda module, input, output: torch.zeros_like(output)

def calculate_head_impacts(model, tokenizer, name='model'):
    is_distilbert = isinstance(model, DistilBertForMaskedLM)
    logits, inputs, mask_token_index = get_logits(model, tokenizer, texts)
    baseline_diff = compute_logit_diff(logits, inputs.input_ids, mask_token_index, tokenizer)
    num_layers = model.config.num_hidden_layers
    num_heads = model.config.num_attention_heads
    impacts = []
    for layer in range(num_layers):
        for head in range(num_heads):
            module = (model.distilbert.transformer.layer[layer].attention
                      if is_distilbert
                      else model.bert.encoder.layer[layer].attention.self)
            hook = module.register_forward_hook(get_ablation_hook(head))
            logits_ablated, _, _ = get_logits(model, tokenizer, texts)
            ablated_diff = compute_logit_diff(logits_ablated, inputs.input_ids, mask_token_index, tokenizer)
            hook.remove()
            impact = 100 * (1 - ablated_diff / baseline_diff) if baseline_diff != 0 else 0
            impacts.append(((layer, head), abs(impact)))
    return impacts

def calculate_mlp_impacts(model, tokenizer, name='model'):
    is_distilbert = isinstance(model, DistilBertForMaskedLM)
    logits, inputs, mask_token_index = get_logits(model, tokenizer, texts)
    baseline_diff = compute_logit_diff(logits, inputs.input_ids, mask_token_index, tokenizer)
    impacts = []
    for layer in range(model.config.num_hidden_layers):
        module = (model.distilbert.transformer.layer[layer].ffn.lin1
                  if is_distilbert
                  else model.bert.encoder.layer[layer].intermediate.dense)
        hook = module.register_forward_hook(get_mlp_ablation_hook())
        try:
            logits_ablated, _, _ = get_logits(model, tokenizer, texts)
            ablated_diff = compute_logit_diff(logits_ablated, inputs.input_ids, mask_token_index, tokenizer)
            impact = 100 * (1 - ablated_diff / baseline_diff) if baseline_diff != 0 else 0
            impacts.append((layer, abs(impact)))
        finally:
            hook.remove()
    return impacts

def get_head_vectors(model, tokenizer, top_heads: List[Tuple[int, int]]):
    is_distilbert = isinstance(model, DistilBertForMaskedLM)
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True, return_dict=True)
    attentions = outputs.attentions
    vectors = {}
    for layer, head in top_heads:
        tensor = attentions[layer][:, head]
        vectors[(layer, head)] = tensor.mean(dim=0)
    return vectors


def get_mlp_vectors(model, tokenizer, top_layers: List[int]):
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True, return_dict=True)
    hidden_states = outputs.hidden_states
    vectors = {}
    for layer in top_layers:
        mlp_output = hidden_states[layer + 1]
        vectors[layer] = mlp_output.mean(dim=(0, 1))
    return vectors

import torch.nn.functional as F

def compute_best_matches_attn(src_dict: Dict, tgt_dict: Dict, threshold=0.0) -> Dict:
    matches = {}
    for s_key, s_mat in src_dict.items():
        best_sim, best_key = -1, None
        for t_key, t_mat in tgt_dict.items():
            sim = torch.mean(F.cosine_similarity(s_mat.unsqueeze(1), t_mat.unsqueeze(1), dim=2)).item()
            if sim > best_sim:
                best_sim, best_key = sim, t_key
        if best_sim >= threshold:
            matches[s_key] = (best_key, best_sim)
    return matches

import numpy as np
from scipy.linalg import svd
from sklearn.metrics.pairwise import cosine_similarity

def activation_eigenvector_similarity(acts1, acts2, k=3):
    acts1_flat = acts1.reshape(-1, acts1.shape[-1])
    acts2_flat = acts2.reshape(-1, acts2.shape[-1])

    cov1 = np.dot(acts1_flat.T, acts1_flat) / acts1_flat.shape[0]
    cov2 = np.dot(acts2_flat.T, acts2_flat) / acts2_flat.shape[0]

    U1, _, _ = svd(cov1)
    U2, _, _ = svd(cov2)
    U1_topk = U1[:, :k]
    U2_topk = U2[:, :k]

    sim_matrix = cosine_similarity(U1_topk.T, U2_topk.T)
    return np.mean(sim_matrix)

def compute_best_matches_mlp(src_dict: Dict, tgt_dict: Dict, threshold=0.0) -> Dict:
    matches = {}
    for s_key, s_vec in src_dict.items():
        best_sim, best_key = -1, None
        for t_key, t_vec in tgt_dict.items():
            sim = activation_eigenvector_similarity(s_vec.cpu().numpy(), t_vec.cpu().numpy())
            if sim > best_sim:
                best_sim, best_key = sim, t_key
        if best_sim >= threshold:
            matches[s_key] = (best_key, best_sim)
    return matches


def compute_best_matches_attn_biased_to_teacher(teacher_dict: Dict, student_dict: Dict) -> Dict:
    matches = {}
    for t_key, t_mat in teacher_dict.items():
        best_sim, best_key = -1, None
        for s_key, s_mat in student_dict.items():
            sim = torch.nn.functional.cosine_similarity(t_mat.flatten(), s_mat.flatten(), dim=0).item()
            if sim > best_sim:
                best_sim, best_key = sim, s_key
        matches[t_key] = (best_key, best_sim)
    return matches

def compute_best_matches_mlp_biased_to_teacher(teacher_dict: Dict, student_dict: Dict) -> Dict:
    matches = {}
    for t_key, t_vec in teacher_dict.items():
        best_sim, best_key = -1, None
        for s_key, s_vec in student_dict.items():
            sim = activation_eigenvector_similarity(t_vec.cpu().numpy(), s_vec.cpu().numpy())
            if sim > best_sim:
                best_sim, best_key = sim, s_key
        matches[t_key] = (best_key, best_sim)
    return matches

def compute_alignment_score(pairs: List[Tuple[float, float, float]]) -> float:
    return sum(S * (1 - abs(I_S - I_T)) for I_S, I_T, S in pairs) / len(pairs)

teacher_heads = calculate_head_impacts(teacher, tokenizer_t, name="Teacher")
student_heads = calculate_head_impacts(student, tokenizer_s, name="Student")
teacher_mlp = calculate_mlp_impacts(teacher, tokenizer_t, name="Teacher")
student_mlp = calculate_mlp_impacts(student, tokenizer_s, name="Student")

influence_teacher = {k: abs(v) for k, v in dict(teacher_heads + teacher_mlp).items()}
influence_student = {k: abs(v) for k, v in dict(student_heads + student_mlp).items()}

teacher_head_vecs = get_head_vectors(teacher, tokenizer_t, [x[0] for x in teacher_heads])
student_head_vecs = get_head_vectors(student, tokenizer_s, [x[0] for x in student_heads])

teacher_mlp_vecs = get_mlp_vectors(teacher, tokenizer_t, [x[0] for x in teacher_mlp])
student_mlp_vecs = get_mlp_vectors(student, tokenizer_s, [x[0] for x in student_mlp])

attn_matches = compute_best_matches_attn_biased_to_teacher(teacher_head_vecs, student_head_vecs)
mlp_matches = compute_best_matches_mlp_biased_to_teacher(teacher_mlp_vecs, student_mlp_vecs)

matches = {**attn_matches, **mlp_matches}
all_pairs = []
for teacher_comp, (student_comp, sim_score) in matches.items():
    I_T = influence_teacher.get(teacher_comp, 0)
    I_S = influence_student.get(student_comp, 0)
    all_pairs.append((I_S, I_T, sim_score))

scaled = [(I_S * len(all_pairs), I_T * len(all_pairs), S) for (I_S, I_T, S) in all_pairs]
sum_I_S = sum(x[0] for x in scaled)
sum_I_T = sum(x[1] for x in scaled)
normalized = [(I_S / sum_I_S, I_T / sum_I_T, S) for (I_S, I_T, S) in scaled]

alignment_score = compute_alignment_score(normalized)

print(f"Alignment score: {alignment_score:.4f}")

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


BERT logit difference: 1.1650
DistilBERT logit difference: 0.4904


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
BertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers

Alignment score: 0.8214


In [ ]:
results = compare_teacher_student(
    teacher_heads,
    student_heads,
    [],
    [],
    teacher_params=85_000_000,
    student_params=42_000_000,
)

import pprint, json
pprint.pprint(results, compact=True)
json.dump(results, open("gpt2_compression_robustness.json", "w"), indent=2)


{'compression_C': 0.5058823529411764,
 'robustness_slope_pp_per_C': 29.65080334476747,
 'student': {'heads': {'gini': 0.6485286587034895,
                       'mean': 15.980501961782059,
                       'median': 6.65235504235433,
                       'single_worst': 142.14585656326383,
                       'worst_k_mean': 89.72786671479419},
             'mean_all': 15.980501961782059,
             'mlp': {'gini': 0.0,
                     'mean': 0.0,
                     'median': 0.0,
                     'single_worst': 0.0,
                     'worst_k_mean': 0.0}},
 'teacher': {'heads': {'gini': 0.6651328675263692,
                       'mean': 0.980683799134985,
                       'median': 0.45320983863768594,
                       'single_worst': 13.423155881020799,
                       'worst_k_mean': 7.044046951082995},
             'mean_all': 0.980683799134985,
             'mlp': {'gini': 0.0,
                     'mean': 0.0,
                     '

## IOI Task:

In [2]:
import torch
import numpy as np
from transformers import AutoTokenizer, BertForMaskedLM, DistilBertForMaskedLM
from typing import List, Tuple, Dict
from scipy.linalg import svd
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine_similarity

device = "cuda" if torch.cuda.is_available() else "cpu"

bert_model = BertForMaskedLM.from_pretrained("bert-base-uncased").to(device)
distilbert_model = DistilBertForMaskedLM.from_pretrained("distilbert-base-uncased").to(device)
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
distilbert_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
bert_model.eval()
distilbert_model.eval()

import re

raw_data = ['After Mark and Sara went to the office, Mark gave a kiss to Sara', 'Then, Brittany and Bryan had a lot of fun at the house. Brittany gave a snack to Bryan', 'Then, Emily and Kimberly went to the station. Kimberly gave a bone to Emily', 'Then, Christine and Stephanie went to the hospital. Christine gave a snack to Stephanie', 'When Emily and Christine got a necklace at the hospital, Christine decided to give it to Emily', 'Then, Megan and Robert were working at the station. Robert decided to give a basketball to Megan', 'Then, Jacob and Tyler were thinking about going to the house. Tyler wanted to give a necklace to Jacob', 'Then, Erin and Lauren had a lot of fun at the station. Lauren gave a basketball to Erin', 'After Brittany and Kevin went to the hospital, Brittany gave a ring to Kevin', 'Then, James and Nicole had a lot of fun at the house. James gave a snack to Nicole', 'Then, Adam and Bradley had a lot of fun at the station. Adam gave a snack to Bradley', 'When Jessica and Robert got a drink at the school, Robert decided to give it to Jessica', 'When Rachel and Megan got a snack at the office, Rachel decided to give it to Megan', 'Then, Danielle and Nathan had a lot of fun at the office. Danielle gave a necklace to Nathan', 'When Steven and Rebecca got a basketball at the hospital, Steven decided to give it to Rebecca', 'Then, Courtney and Nicole went to the garden. Nicole gave a bone to Courtney', 'After Heather and Alicia went to the hospital, Alicia gave a computer to Heather', 'Then, Nicole and Christina went to the office. Christina gave a bone to Nicole', 'Then, Shannon and Christopher were working at the restaurant. Shannon decided to give a computer to Christopher', 'Then, Joshua and Paul had a lot of fun at the restaurant. Paul gave a snack to Joshua', 'Then, Christopher and Scott went to the station. Scott gave a snack to Christopher', 'When Brittany and Brian got a snack at the hospital, Brian decided to give it to Brittany', 'Then, Daniel and Tiffany went to the garden. Tiffany gave a ring to Daniel', 'Then, Tyler and Sarah had a long argument, and afterwards Tyler said to Sarah', 'Then, Angela and Bradley went to the school. Bradley gave a drink to Angela', 'When Aaron and Kyle got a computer at the house, Kyle decided to give it to Aaron', 'Then, Amanda and Travis were thinking about going to the garden. Amanda wanted to give a basketball to Travis', 'Then, Heather and Kevin had a long argument, and afterwards Kevin said to Heather', 'After Stephanie and Paul went to the restaurant, Paul gave a ring to Stephanie', 'Then, Katie and Charles went to the station. Katie gave a computer to Charles', 'Then, Elizabeth and Timothy had a lot of fun at the hospital. Timothy gave a snack to Elizabeth', 'Then, Jesse and Elizabeth went to the school. Elizabeth gave a drink to Jesse', 'Then, Dustin and Eric went to the restaurant. Eric gave a computer to Dustin', 'Then, Justin and David were working at the restaurant. David decided to give a computer to Justin', 'Then, Tyler and Cody were thinking about going to the station. Tyler wanted to give a kiss to Cody', 'When Katie and Kyle got a computer at the house, Kyle decided to give it to Katie', 'Then, Erin and Dustin had a lot of fun at the garden. Erin gave a ring to Dustin', 'Then, Lauren and Alicia went to the hospital. Lauren gave a ring to Alicia', 'Then, Elizabeth and Alexander went to the store. Alexander gave a computer to Elizabeth', 'Then, Bradley and Jennifer were thinking about going to the restaurant. Bradley wanted to give a snack to Jennifer', 'Then, Aaron and Lindsey had a lot of fun at the restaurant. Aaron gave a drink to Lindsey', 'Then, Sean and Robert had a long argument, and afterwards Robert said to Sean', 'After Jose and Lauren went to the restaurant, Jose gave a necklace to Lauren', 'Then, Erin and Katie went to the house. Erin gave a ring to Katie', 'When Robert and Laura got a kiss at the office, Laura decided to give it to Robert', 'Then, Tiffany and Kelly were thinking about going to the station. Kelly wanted to give a drink to Tiffany', 'Then, Nicole and Samantha had a lot of fun at the station. Samantha gave a snack to Nicole', 'Then, Joshua and Anthony were thinking about going to the restaurant. Anthony wanted to give a computer to Joshua', 'When Lindsay and Sean got a basketball at the store, Sean decided to give it to Lindsay', 'When Rachel and William got a drink at the store, Rachel decided to give it to William', 'Then, Eric and Andrew went to the garden. Andrew gave a bone to Eric', 'When Vanessa and Christina got a drink at the hospital, Christina decided to give it to Vanessa', 'Then, Patrick and Stephen were working at the station. Stephen decided to give a bone to Patrick', 'Then, Gregory and Nicole went to the restaurant. Gregory gave a kiss to Nicole', 'Then, Benjamin and Samuel were working at the hospital. Benjamin decided to give a ring to Samuel', 'Then, Kyle and Courtney were working at the station. Kyle decided to give a bone to Courtney', 'When Sara and Patrick got a drink at the office, Patrick decided to give it to Sara', 'When Lauren and Nicole got a drink at the school, Nicole decided to give it to Lauren', 'After Jeremy and Jennifer went to the station, Jennifer gave a drink to Jeremy', 'Then, Lindsey and Kristen went to the station. Lindsey gave a ring to Kristen', 'Then, Andrea and Jamie had a long argument, and afterwards Andrea said to Jamie', 'Then, Nathan and Rebecca went to the restaurant. Rebecca gave a drink to Nathan', 'Then, Lindsey and Sarah were working at the station. Sarah decided to give a computer to Lindsey', 'After Katie and Travis went to the garden, Katie gave a necklace to Travis', 'Then, Alexander and Sara were working at the office. Sara decided to give a snack to Alexander', 'Then, Dustin and Ashley were working at the garden. Ashley decided to give a basketball to Dustin', 'Then, Jesse and Timothy had a long argument, and afterwards Timothy said to Jesse', 'Then, Bradley and Kenneth went to the garden. Kenneth gave a snack to Bradley', 'Then, Kyle and Katie had a long argument, and afterwards Kyle said to Katie', 'Then, Amanda and Christina were thinking about going to the office. Christina wanted to give a drink to Amanda', 'Then, Nathan and Megan had a long argument, and afterwards Nathan said to Megan', 'Then, Mary and James had a long argument, and afterwards Mary said to James', 'Then, Elizabeth and Justin were thinking about going to the hospital. Elizabeth wanted to give a snack to Justin', 'Then, Christopher and Jose were working at the hospital. Christopher decided to give a bone to Jose', 'Then, Aaron and Danielle went to the hospital. Danielle gave a drink to Aaron', 'When Rebecca and Melissa got a drink at the school, Rebecca decided to give it to Melissa', 'Then, Patrick and Joseph were working at the house. Joseph decided to give a necklace to Patrick', 'Then, Kristen and Jessica were thinking about going to the hospital. Kristen wanted to give a ring to Jessica', 'Then, Cody and Jesse were thinking about going to the station. Cody wanted to give a computer to Jesse', 'Then, Elizabeth and Erica were thinking about going to the school. Erica wanted to give a basketball to Elizabeth', 'Then, Alexander and Melissa were thinking about going to the restaurant. Alexander wanted to give a kiss to Melissa', 'Then, Bryan and Jennifer had a long argument, and afterwards Jennifer said to Bryan', 'Then, Christine and Kimberly were thinking about going to the garden. Christine wanted to give a computer to Kimberly', 'After Benjamin and Cody went to the school, Benjamin gave a bone to Cody', 'Then, Dustin and Kimberly were thinking about going to the hospital. Kimberly wanted to give a bone to Dustin', 'Then, Christina and Patrick were working at the office. Christina decided to give a kiss to Patrick', 'When David and Justin got a drink at the school, Justin decided to give it to David', 'Then, Angela and Kimberly were working at the house. Kimberly decided to give a ring to Angela', 'Then, Ryan and Sara were working at the office. Ryan decided to give a computer to Sara', 'Then, Kenneth and Amy were thinking about going to the garden. Kenneth wanted to give a basketball to Amy', 'When Matthew and Travis got a basketball at the office, Travis decided to give it to Matthew', 'Then, Daniel and Kenneth had a long argument, and afterwards Kenneth said to Daniel', 'Then, Jason and Jennifer went to the hospital. Jennifer gave a snack to Jason', 'Then, John and Adam went to the restaurant. Adam gave a drink to John', 'When Jeffrey and Jose got a bone at the house, Jose decided to give it to Jeffrey', 'Then, John and Lindsey went to the house. John gave a computer to Lindsey', 'Then, Katie and Jeffrey had a lot of fun at the station. Jeffrey gave a snack to Katie', 'After Kimberly and John went to the office, John gave a necklace to Kimberly', 'Then, Michelle and Thomas had a lot of fun at the garden. Thomas gave a snack to Michelle', 'Then, Kristen and Heather went to the store. Heather gave a ring to Kristen']


texts = []
targets = []
distractors = []

pattern = re.compile(
    r"(?i)(?:After|Then|When)?[^.,;]*?\b(\w+)\b and \b(\w+)\b.*?"
    r"\b(\1|\2)\b.*?"
    r"(?:gave|wanted to give|decided to give|said to).*?\bto (\w+)\b"
)

for sentence in raw_data:
    match = pattern.search(sentence)
    if match:
        person1, person2, giver, recipient = match.groups()

        recipient_pattern = re.compile(rf"\bto {re.escape(recipient)}\b", re.IGNORECASE)
        masked_sentence = recipient_pattern.sub("to [MASK]", sentence, count=1)

        texts.append(masked_sentence.strip().rstrip('.') + '.')
        targets.append(recipient.lower())
        distractors.append((person1 if recipient.lower() != person1.lower() else person2).lower())

def compute_logit_diff_ioi(model, tokenizer, texts, targets, distractors):
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True).to(device)
    mask_token_index = (inputs.input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)
    with torch.no_grad():
        logits = model(**inputs).logits
    diffs = []
    for i, (b, pos) in enumerate(zip(*mask_token_index)):
        tgt_id = tokenizer.convert_tokens_to_ids(targets[i])
        dst_id = tokenizer.convert_tokens_to_ids(distractors[i])
        logits_slice = logits[b, pos]
        diffs.append((logits_slice[tgt_id] - logits_slice[dst_id]).item())
    return np.mean(diffs)

bert_diff = compute_logit_diff_ioi(bert_model, bert_tokenizer, texts, targets, distractors)
distilbert_diff = compute_logit_diff_ioi(distilbert_model, distilbert_tokenizer, texts, targets, distractors)

print(f"BERT logit difference: {bert_diff:.4f}")
print(f"DistilBERT logit difference: {distilbert_diff:.4f}")

def get_logits(model, tokenizer, texts):
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True).to(device)
    mask_token_index = (inputs.input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)
    with torch.no_grad():
        outputs = model(**inputs, return_dict=True)
    return outputs.logits, inputs, mask_token_index

def compute_logit_diff_ioi_logits(logits, input_ids, mask_token_index, tokenizer, targets, distractors):
    diffs = []
    for i, (b, pos) in enumerate(zip(*mask_token_index)):
        tgt_id = tokenizer.convert_tokens_to_ids(targets[i])
        dst_id = tokenizer.convert_tokens_to_ids(distractors[i])
        logits_slice = logits[b, pos]
        diffs.append((logits_slice[tgt_id] - logits_slice[dst_id]).item())
    return np.mean(diffs)

def get_ablation_hook(head_idx):
    def ablate_head(module, input, output):
        if isinstance(output, tuple):
            attn_output = output[0]
            attn_output[:, head_idx] = 0
            return (attn_output,) + output[1:]
        output[:, head_idx] = 0
        return output
    return ablate_head

def get_mlp_ablation_hook():
    return lambda module, input, output: torch.zeros_like(output)

def calculate_head_impacts(model, tokenizer, texts, targets, distractors):
    is_distilbert = isinstance(model, DistilBertForMaskedLM)
    logits, inputs, mask_token_index = get_logits(model, tokenizer, texts)
    baseline_diff = compute_logit_diff_ioi_logits(logits, inputs.input_ids, mask_token_index, tokenizer, targets, distractors)
    num_layers = model.config.num_hidden_layers
    num_heads = model.config.num_attention_heads
    impacts = []
    for layer in range(num_layers):
        for head in range(num_heads):
            module = model.distilbert.transformer.layer[layer].attention if is_distilbert else model.bert.encoder.layer[layer].attention.self
            hook = module.register_forward_hook(get_ablation_hook(head))
            logits_ablated, _, _ = get_logits(model, tokenizer, texts)
            ablated_diff = compute_logit_diff_ioi_logits(logits_ablated, inputs.input_ids, mask_token_index, tokenizer, targets, distractors)
            hook.remove()
            impact = 100 * (1 - ablated_diff / baseline_diff) if baseline_diff != 0 else 0
            impacts.append(((layer, head), abs(impact)))
    return impacts

def calculate_mlp_impacts(model, tokenizer, texts, targets, distractors):
    is_distilbert = isinstance(model, DistilBertForMaskedLM)
    logits, inputs, mask_token_index = get_logits(model, tokenizer, texts)
    baseline_diff = compute_logit_diff_ioi_logits(logits, inputs.input_ids, mask_token_index, tokenizer, targets, distractors)
    impacts = []
    for layer in range(model.config.num_hidden_layers):
        module = model.distilbert.transformer.layer[layer].ffn.lin1 if is_distilbert else model.bert.encoder.layer[layer].intermediate.dense
        hook = module.register_forward_hook(get_mlp_ablation_hook())
        try:
            logits_ablated, _, _ = get_logits(model, tokenizer, texts)
            ablated_diff = compute_logit_diff_ioi_logits(logits_ablated, inputs.input_ids, mask_token_index, tokenizer, targets, distractors)
            impact = 100 * (1 - ablated_diff / baseline_diff) if baseline_diff != 0 else 0
            impacts.append((layer, abs(impact)))
        finally:
            hook.remove()
    return impacts

def get_head_vectors(model, tokenizer, texts, top_heads: List[Tuple[int, int]]):
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True, return_dict=True)
    attentions = outputs.attentions
    vectors = {}
    for layer, head in top_heads:
        vectors[(layer, head)] = attentions[layer][:, head].mean(dim=0)
    return vectors

def get_mlp_vectors(model, tokenizer, texts, top_layers: List[int]):
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True, return_dict=True)
    hidden_states = outputs.hidden_states
    vectors = {}
    for layer in top_layers:
        vectors[layer] = hidden_states[layer + 1].mean(dim=(0, 1))
    return vectors

def activation_eigenvector_similarity(acts1, acts2, k=3):
    acts1_flat = acts1.reshape(-1, acts1.shape[-1])
    acts2_flat = acts2.reshape(-1, acts2.shape[-1])
    cov1 = np.dot(acts1_flat.T, acts1_flat) / acts1_flat.shape[0]
    cov2 = np.dot(acts2_flat.T, acts2_flat) / acts2_flat.shape[0]
    U1, _, _ = svd(cov1)
    U2, _, _ = svd(cov2)
    sim_matrix = sklearn_cosine_similarity(U1[:, :k].T, U2[:, :k].T)
    return np.mean(sim_matrix)

def compute_best_matches_attn(src_dict: Dict, tgt_dict: Dict) -> Dict:
    matches = {}
    for s_key, s_mat in src_dict.items():
        best_sim, best_key = -1, None
        for t_key, t_mat in tgt_dict.items():
            sim = torch.nn.functional.cosine_similarity(s_mat.flatten(), t_mat.flatten(), dim=0).item()
            if sim > best_sim:
                best_sim, best_key = sim, t_key
        matches[s_key] = (best_key, best_sim)
    return matches

def compute_best_matches_mlp(src_dict: Dict, tgt_dict: Dict) -> Dict:
    matches = {}
    for s_key, s_vec in src_dict.items():
        best_sim, best_key = -1, None
        for t_key, t_vec in tgt_dict.items():
            sim = activation_eigenvector_similarity(s_vec.cpu().numpy(), t_vec.cpu().numpy())
            if sim > best_sim:
                best_sim, best_key = sim, t_key
        matches[s_key] = (best_key, best_sim)
    return matches


def compute_best_matches_attn_biased_to_teacher(teacher_dict: Dict, student_dict: Dict) -> Dict:
    matches = {}
    for t_key, t_mat in teacher_dict.items():
        best_sim, best_key = -1, None
        for s_key, s_mat in student_dict.items():
            sim = torch.nn.functional.cosine_similarity(t_mat.flatten(), s_mat.flatten(), dim=0).item()
            if sim > best_sim:
                best_sim, best_key = sim, s_key
        matches[t_key] = (best_key, best_sim)
    return matches

def compute_best_matches_mlp_biased_to_teacher(teacher_dict: Dict, student_dict: Dict) -> Dict:
    matches = {}
    for t_key, t_vec in teacher_dict.items():
        best_sim, best_key = -1, None
        for s_key, s_vec in student_dict.items():
            sim = activation_eigenvector_similarity(t_vec.cpu().numpy(), s_vec.cpu().numpy())
            if sim > best_sim:
                best_sim, best_key = sim, s_key
        matches[t_key] = (best_key, best_sim)
    return matches


def compute_alignment_score(pairs: List[Tuple[float, float, float]]) -> float:
    return sum(S * (1 - abs(I_S - I_T)) for I_S, I_T, S in pairs) / len(pairs)

bert_diff = compute_logit_diff_ioi(bert_model, bert_tokenizer, texts, targets, distractors)
distilbert_diff = compute_logit_diff_ioi(distilbert_model, distilbert_tokenizer, texts, targets, distractors)

teacher_heads = calculate_head_impacts(bert_model, bert_tokenizer, texts, targets, distractors)
student_heads = calculate_head_impacts(distilbert_model, distilbert_tokenizer, texts, targets, distractors)
teacher_mlp = calculate_mlp_impacts(bert_model, bert_tokenizer, texts, targets, distractors)
student_mlp = calculate_mlp_impacts(distilbert_model, distilbert_tokenizer, texts, targets, distractors)

influence_teacher = {k: v for k, v in dict(teacher_heads + teacher_mlp).items()}
influence_student = {k: v for k, v in dict(student_heads + student_mlp).items()}

student_head_vecs = get_head_vectors(distilbert_model, distilbert_tokenizer, texts, [x[0] for x in student_heads])
teacher_head_vecs = get_head_vectors(bert_model, bert_tokenizer, texts, [x[0] for x in teacher_heads])
student_mlp_vecs = get_mlp_vectors(distilbert_model, distilbert_tokenizer, texts, [x[0] for x in student_mlp])
teacher_mlp_vecs = get_mlp_vectors(bert_model, bert_tokenizer, texts, [x[0] for x in teacher_mlp])

attn_matches = compute_best_matches_attn_biased_to_teacher(teacher_head_vecs, student_head_vecs)
mlp_matches = compute_best_matches_mlp_biased_to_teacher(teacher_mlp_vecs, student_mlp_vecs)

matches = {**attn_matches, **mlp_matches}
all_pairs = []
for teacher_comp, (student_comp, sim_score) in matches.items():
    I_T = influence_teacher.get(teacher_comp, 0)
    I_S = influence_student.get(student_comp, 0)
    all_pairs.append((I_S, I_T, sim_score))

scaled = [(I_S * len(all_pairs), I_T * len(all_pairs), S) for (I_S, I_T, S) in all_pairs]
sum_I_S = sum(x[0] for x in scaled)
sum_I_T = sum(x[1] for x in scaled)
normalized = [(I_S / sum_I_S, I_T / sum_I_T, S) for (I_S, I_T, S) in scaled]

alignment_score = compute_alignment_score(normalized)

print(f"Alignment score: {alignment_score:.4f}")


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


BERT logit difference: 5.2719
DistilBERT logit difference: 3.6959


DistilBertSdpaAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.
BertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


Alignment score: 0.8527


In [ ]:
results = compare_teacher_student(
    teacher_heads,
    student_heads,
    [],
    [],
    teacher_params=85_000_000,
    student_params=42_000_000,
)

import pprint, json
pprint.pprint(results, compact=True)
json.dump(results, open("gpt2_compression_robustness.json", "w"), indent=2)


{'compression_C': 0.5058823529411764,
 'robustness_slope_pp_per_C': 0.7914458815134658,
 'student': {'heads': {'gini': 0.700022977540101,
                       'mean': 1.438843285209041,
                       'median': 0.43863020536807906,
                       'single_worst': 11.04566392722136,
                       'worst_k_mean': 9.12185643812059},
             'mean_all': 1.438843285209041,
             'mlp': {'gini': 0.0,
                     'mean': 0.0,
                     'median': 0.0,
                     'single_worst': 0.0,
                     'worst_k_mean': 0.0}},
 'teacher': {'heads': {'gini': 0.6641145002717123,
                       'mean': 1.0384647804434053,
                       'median': 0.34933881032164216,
                       'single_worst': 9.163933911104127,
                       'worst_k_mean': 5.744460737137347},
             'mean_all': 1.0384647804434053,
             'mlp': {'gini': 0.0,
                     'mean': 0.0,
                     '

# GPT2-small / DistilGPT2 Numeral Sequence Completion:

In [ ]:
texts = ['Van done in 1. Hat done in 2. Ring done in 3. Desk done in 4. Sun done in', 'Ice done in 2. Snow done in 3. Watch done in 4. Sun done in 5. Table done in', 'Ring done in 3. Moon done in 4. Queen done in 5. Book done in 6. Rose done in', 'Queen done in 4. Oil done in 5. Rose done in 6. Desk done in 7. Car done in', 'Light done in 5. Arm done in 6. Road done in 7. Book done in 8. Ice done in', 'Ball done in 6. Cow done in 7. Book done in 8. Rose done in 9. Key done in', 'Road done in 7. Key done in 8. Ocean done in 9. Key done in 10. Queen done in', 'House done in 8. Rose done in 9. Key done in 10. Hat done in 11. Van done in', 'Ring done in 1. Car done in 2. Apple done in 3. Pear done in 4. Moon done in', 'Star done in 2. Sun done in 3. Road done in 4. Queen done in 5. Box done in', 'Hill done in 3. Ant done in 4. Apple done in 5. House done in 6. Hat done in', 'Chair done in 4. Van done in 5. Orange done in 6. Queen done in 7. Zip done in', 'Gate done in 5. Desk done in 6. Wolf done in 7. Rain done in 8. Flag done in', 'Queen done in 6. House done in 7. Light done in 8. Cat done in 9. Moon done in', 'Zip done in 7. Car done in 8. Ring done in 9. Hand done in 10. Fish done in', 'Snake done in 8. Queen done in 9. Window done in 10. Ear done in 11. Orange done in', 'Ice done in 1. Sand done in 2. Desk done in 3. Van done in 4. Hat done in', 'Camera done in 2. Watch done in 3. Dog done in 4. Book done in 5. Desk done in', 'Rain done in 3. Dog done in 4. Rat done in 5. Nut done in 6. Ocean done in', 'Night done in 4. Hat done in 5. Wall done in 6. Book done in 7. Tree done in', 'Gate done in 5. Chair done in 6. Ring done in 7. Hill done in 8. Car done in', 'Ax done in 6. Fan done in 7. Cat done in 8. Ring done in 9. Ring done in', 'Rose done in 7. Night done in 8. Van done in 9. Orange done in 10. Apple done in', 'Orange done in 8. Ear done in 9. Book done in 10. Ring done in 11. Bird done in', 'Ball done in 1. Wind done in 2. Wallet done in 3. Tree done in 4. Sand done in', 'Table done in 2. Jam done in 3. Apple done in 4. Pear done in 5. Ice done in', 'Gate done in 3. Train done in 4. Hat done in 5. Fan done in 6. Key done in', 'Window done in 4. Desk done in 5. Year done in 6. Van done in 7. Camera done in', 'Car done in 5. Watch done in 6. Oil done in 7. Queen done in 8. Fan done in', 'Mouse done in 6. Rose done in 7. Table done in 8. Clock done in 9. Queen done in', 'Year done in 7. Wheel done in 8. Wolf done in 9. Arm done in 10. Ax done in', 'Key done in 8. Ring done in 9. Night done in 10. Fan done in 11. Clock done in', 'Glass done in 1. Key done in 2. Nut done in 3. Rat done in 4. Van done in', 'Star done in 2. Pear done in 3. Bug done in 4. Bird done in 5. Ring done in', 'Road done in 3. Wind done in 4. Rat done in 5. Wolf done in 6. Map done in', 'Car done in 4. Ant done in 5. Rose done in 6. Van done in 7. Wolf done in', 'Zip done in 5. Hat done in 6. Wind done in 7. Rain done in 8. Car done in', 'Ring done in 6. Light done in 7. Jar done in 8. Dog done in 9. Fish done in', 'Bug done in 7. Key done in 8. Gate done in 9. Orange done in 10. Ring done in', 'Flag done in 8. Fire done in 9. Pear done in 10. Ax done in 11. Ear done in', 'Ring done in 1. Flag done in 2. Queen done in 3. Map done in 4. Camera done in', 'Desk done in 2. Fire done in 3. Desk done in 4. Road done in 5. Watch done in', 'Queen done in 3. House done in 4. Flag done in 5. Tree done in 6. Ring done in', 'Desk done in 4. Window done in 5. Fish done in 6. Jar done in 7. Hat done in', 'Snake done in 5. Key done in 6. Glass done in 7. Van done in 8. House done in', 'Camera done in 6. Year done in 7. Rose done in 8. Map done in 9. Fire done in', 'Rose done in 7. Snake done in 8. Corn done in 9. Desk done in 10. Zip done in', 'Table done in 8. Key done in 9. Eye done in 10. Van done in 11. Key done in', 'Star done in 1. Hat done in 2. Star done in 3. Eye done in 4. Gold done in', 'Gate done in 2. Queen done in 3. Camera done in 4. Zip done in 5. Ocean done in', 'Fan done in 3. Corn done in 4. Apple done in 5. Van done in 6. Cat done in', 'Key done in 4. Gold done in 5. Star done in 6. Queen done in 7. Arm done in', 'Van done in 5. Snake done in 6. Car done in 7. Star done in 8. Watch done in', 'Jar done in 6. Flag done in 7. Window done in 8. Sun done in 9. Van done in', 'Wolf done in 7. Watch done in 8. Ring done in 9. Desk done in 10. Car done in', 'Cow done in 8. Tree done in 9. Book done in 10. Wall done in 11. Flag done in', 'Van done in 1. Wall done in 2. Ring done in 3. Car done in 4. Key done in', 'Jar done in 2. Wallet done in 3. Fish done in 4. Watch done in 5. Orange done in', 'Watch done in 3. Orange done in 4. Table done in 5. Corn done in 6. Van done in', 'Lake done in 4. Star done in 5. Fan done in 6. Fan done in 7. Night done in', 'Fan done in 5. Ball done in 6. Desk done in 7. Jam done in 8. Tree done in', 'Key done in 6. Jar done in 7. Night done in 8. Fish done in 9. Fan done in', 'Ice done in 7. Rose done in 8. Wall done in 9. Road done in 10. Window done in', 'Tree done in 8. Hill done in 9. Wall done in 10. Queen done in 11. Desk done in', 'Gold done in 1. King done in 2. Queen done in 3. Desk done in 4. Night done in', 'Van done in 2. Sand done in 3. Tree done in 4. Car done in 5. Cow done in', 'Pear done in 3. Eye done in 4. Lake done in 5. Fan done in 6. Light done in', 'Jam done in 4. Road done in 5. Corn done in 6. Ice done in 7. Pen done in', 'Iron done in 5. Ring done in 6. Star done in 7. Van done in 8. Star done in', 'Bird done in 6. Wheel done in 7. Cat done in 8. Desk done in 9. Desk done in', 'Fish done in 7. Gold done in 8. King done in 9. Ax done in 10. Jam done in', 'Desk done in 8. House done in 9. Cow done in 10. Hill done in 11. Gate done in', 'Nut done in 1. Light done in 2. Sand done in 3. Wall done in 4. Key done in', 'Tree done in 2. Fan done in 3. Car done in 4. Desk done in 5. Ring done in', 'Orange done in 3. Ocean done in 4. Sun done in 5. Bug done in 6. Queen done in', 'Orange done in 4. House done in 5. Watch done in 6. Corn done in 7. Ring done in', 'Wind done in 5. Orange done in 6. Rose done in 7. Zip done in 8. Queen done in', 'Iron done in 6. Cat done in 7. Glass done in 8. Box done in 9. Rose done in', 'Clock done in 7. Zip done in 8. Cat done in 9. Snow done in 10. Ant done in', 'Rose done in 8. Ear done in 9. Rose done in 10. Rat done in 11. Orange done in', 'Road done in 1. Ring done in 2. Pen done in 3. Oil done in 4. Camera done in', 'House done in 2. Zip done in 3. Queen done in 4. Snow done in 5. Lake done in', 'Van done in 3. Tree done in 4. Wall done in 5. Rat done in 6. King done in', 'Ball done in 4. Hill done in 5. Clock done in 6. Ear done in 7. Van done in', 'Night done in 5. Ice done in 6. Apple done in 7. Tree done in 8. House done in', 'Clock done in 6. Orange done in 7. Queen done in 8. Desk done in 9. Arm done in', 'Map done in 7. Sun done in 8. Clock done in 9. Cat done in 10. Ball done in', 'Desk done in 8. Key done in 9. Star done in 10. Cow done in 11. Rose done in', 'Ring done in 1. Fan done in 2. Window done in 3. Tree done in 4. Chair done in', 'House done in 2. Nut done in 3. Queen done in 4. Zip done in 5. Rose done in', 'Window done in 3. Gate done in 4. Zip done in 5. Key done in 6. Snow done in', 'Corn done in 4. Ice done in 5. Eye done in 6. Light done in 7. Desk done in', 'Hill done in 5. Snake done in 6. Orange done in 7. Road done in 8. Flag done in', 'Queen done in 6. Window done in 7. Zip done in 8. Cow done in 9. Fire done in', 'Star done in 7. Hat done in 8. Chair done in 9. Car done in 10. Ring done in', 'Desk done in 8. Watch done in 9. Desk done in 10. Ax done in 11. Queen done in', 'Train done in 1. Snow done in 2. Cow done in 3. Ring done in 4. Queen done in', 'Window done in 2. Pen done in 3. Ax done in 4. Iron done in 5. Nut done in', 'Desk done in 3. Flag done in 4. Fan done in 5. Hat done in 6. Chair done in', 'King done in 4. Star done in 5. Cow done in 6. Wallet done in 7. Fan done in', 'Star done in 5. Watch done in 6. Car done in 7. Van done in 8. Queen done in']

import torch
import numpy as np
import re
from transformers import AutoTokenizer, BertForMaskedLM, DistilBertForMaskedLM, GPT2LMHeadModel
device = "cuda" if torch.cuda.is_available() else "cpu"
teacher_str = "gpt2"
student_str = "distilgpt2"

teacher_model = GPT2LMHeadModel.from_pretrained(teacher_str).to(device)
teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_str)
teacher_tokenizer.pad_token = teacher_tokenizer.eos_token
teacher_tokenizer.padding_side = "left"

student_model = GPT2LMHeadModel.from_pretrained(student_str).to(device)
student_tokenizer = AutoTokenizer.from_pretrained(student_str)
student_tokenizer.pad_token = student_tokenizer.eos_token
student_tokenizer.padding_side = "left"

teacher_model.eval()
student_model.eval()

def normalize_numerals(texts):
    normalized_texts = []
    for text in texts:
        matches = list(re.finditer(r'\b\d+\b', text))
        mapping = {match.group(): str(i + 1) for i, match in enumerate(matches)}
        normalized_text = re.sub(r'\b\d+\b', lambda m: mapping[m.group()], text)
        normalized_texts.append(normalized_text)
    return normalized_texts

texts = normalize_numerals(texts)

def compute_logit_diff(model, tokenizer, texts, target_token, distractor_token):
    diffs = []
    target_id = tokenizer.encode(" " + target_token, add_special_tokens=False)[0]
    distractor_id = tokenizer.encode(" " + distractor_token, add_special_tokens=False)[0]

    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
        # Get the logits for the last token
        last_token_logits = logits[0, -1, :]
        diff = last_token_logits[target_id] - last_token_logits[distractor_id]
        diffs.append(diff.item())

    return np.mean(diffs)

target_token = '5'
distractor_token = '4'
teacher_diff = compute_logit_diff(teacher_model, teacher_tokenizer, texts, target_token, distractor_token)
student_diff = compute_logit_diff(student_model, student_tokenizer, texts, target_token, distractor_token)

print(f"Teacher logit difference: {teacher_diff:.4f}")
print(f"Student logit difference: {student_diff:.4f}")

import torch
import numpy as np
import re
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict
from torch.nn.functional import cosine_similarity
from transformers import (AutoTokenizer, GPT2LMHeadModel)

device = "cuda" if torch.cuda.is_available() else "cpu"
teacher_str = "gpt2"
student_str = "distilgpt2"

teacher = GPT2LMHeadModel.from_pretrained(teacher_str).to(device)
student = GPT2LMHeadModel.from_pretrained(student_str).to(device)
tokenizer_t = AutoTokenizer.from_pretrained(teacher_str)
tokenizer_t.pad_token = tokenizer_t.eos_token
tokenizer_t.padding_side = "left"

tokenizer_s = AutoTokenizer.from_pretrained(student_str)
tokenizer_s.pad_token = tokenizer_s.eos_token
tokenizer_s.padding_side = "left"


teacher.eval()
student.eval()

def normalize_numerals(texts):
    normalized_texts = []
    for text in texts:
        matches = list(re.finditer(r'\b\d+\b', text))
        mapping = {match.group(): str(i + 1) for i, match in enumerate(matches)}
        normalized_text = re.sub(r'\b\d+\b', lambda m: mapping[m.group()], text)
        normalized_texts.append(normalized_text)
    return normalized_texts

texts = normalize_numerals(texts)

def get_logits(model, tokenizer, texts):
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs, return_dict=True)
    # Get the logits for the last token
    last_token_logits = outputs.logits[:, -1, :]
    return last_token_logits, inputs

def compute_logit_diff(logits, input_ids, tokenizer):
    diffs = []
    target_id = tokenizer.encode(" " + target_token, add_special_tokens=False)[0]
    distractor_id = tokenizer.encode(" " + distractor_token, add_special_tokens=False)[0]

    for i in range(logits.shape[0]):
        logits_slice = logits[i, :]
        correct_logit = logits_slice[target_id]
        incorrect_logit = logits_slice[distractor_id]
        diffs.append((correct_logit - incorrect_logit).item())
    return np.mean(diffs)

def get_ablation_hook(head_idx):
    def ablate_head(module, input, output):
        if isinstance(output, tuple):
            attn_output = output[0]
            # Ablate the specified head
            attn_output[:, head_idx] = 0
            return (attn_output,) + output[1:]
        # If output is not a tuple, it's likely the attention output directly
        output[:, head_idx] = 0
        return output
    return ablate_head

def get_mlp_ablation_hook():
    return lambda module, input, output: torch.zeros_like(output)

def calculate_head_impacts(model, tokenizer, texts, target_token, distractor_token):
    logits, inputs = get_logits(model, tokenizer, texts)
    baseline_diff = compute_logit_diff(logits, inputs.input_ids, tokenizer)
    num_layers = model.config.n_layer
    num_heads = model.config.n_head
    impacts = []

    for layer in range(num_layers):
        # Assuming the attention module is in model.transformer.h[layer].attn
        module = model.transformer.h[layer].attn
        for head in range(num_heads):
            hook = module.register_forward_hook(get_ablation_hook(head))
            logits_ablated, _ = get_logits(model, tokenizer, texts)
            ablated_diff = compute_logit_diff(logits_ablated, inputs.input_ids, tokenizer)
            hook.remove()
            impact = 100 * (1 - ablated_diff / baseline_diff) if baseline_diff != 0 else 0
            impacts.append(((layer, head), abs(impact)))
    return impacts

def calculate_mlp_impacts(model, tokenizer, texts, target_token, distractor_token):
    logits, inputs = get_logits(model, tokenizer, texts)
    baseline_diff = compute_logit_diff(logits, inputs.input_ids, tokenizer)
    impacts = []
    for layer in range(model.config.n_layer):
        # Assuming the MLP module is in model.transformer.h[layer].mlp.c_fc
        module = model.transformer.h[layer].mlp.c_fc
        hook = module.register_forward_hook(get_mlp_ablation_hook())
        try:
            logits_ablated, _ = get_logits(model, tokenizer, texts)
            ablated_diff = compute_logit_diff(logits_ablated, inputs.input_ids, tokenizer)
            impact = 100 * (1 - ablated_diff / baseline_diff) if baseline_diff != 0 else 0
            impacts.append((layer, abs(impact)))
        finally:
            hook.remove()
    return impacts


def get_head_vectors(model, tokenizer, texts, top_heads: List[Tuple[int, int]]):
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True, return_dict=True)
    attentions = outputs.attentions
    vectors = {}
    for layer, head in top_heads:
        # Get attention weights for the last token across all input tokens
        # Shape: (batch_size, num_heads, seq_len, seq_len)
        # We want the attention from the last token (query) to all previous tokens (keys)
        # So we need attentions[layer][:, head, -1, :]
        # Take the mean across the batch
        vectors[(layer, head)] = attentions[layer][:, head, -1, :].mean(dim=0)
    return vectors


def get_mlp_vectors(model, tokenizer, texts, top_layers: List[int]):
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True, return_dict=True)
    hidden_states = outputs.hidden_states
    vectors = {}
    for layer in top_layers:
        # Get the MLP output for the last token
        # Shape of hidden_states is (layer, batch_size, seq_len, hidden_size)
        # We want hidden_states[layer+1] for the output of layer 'layer'
        # We need the output at the last token position: hidden_states[layer + 1][:, -1, :]
        # Take the mean across the batch
        mlp_output = hidden_states[layer + 1][:, -1, :]
        vectors[layer] = mlp_output.mean(dim=0)
    return vectors

import torch.nn.functional as F

def compute_best_matches_attn_biased_to_teacher(teacher_dict: Dict, student_dict: Dict) -> Dict:
    matches = {}
    for t_key, t_vec in teacher_dict.items():
        best_sim, best_key = -1, None
        for s_key, s_vec in student_dict.items():
            # Adjust for potential sequence length differences
            min_len = min(t_vec.shape[-1], s_vec.shape[-1])
            sim = torch.nn.functional.cosine_similarity(t_vec[-min_len:], s_vec[-min_len:], dim=0).item()
            if sim > best_sim:
                best_sim, best_key = sim, s_key
        matches[t_key] = (best_key, best_sim)
    return matches


def compute_best_matches_mlp_biased_to_teacher(teacher_dict: Dict, student_dict: Dict) -> Dict:
    matches = {}
    for t_key, t_vec in teacher_dict.items():
        best_sim, best_key = -1, None
        for s_key, s_vec in student_dict.items():
             # MLP vectors are already meaned over sequence length, so direct comparison is fine
            sim = torch.nn.functional.cosine_similarity(t_vec, s_vec, dim=0).item()
            if sim > best_sim:
                best_sim, best_key = sim, s_key
        matches[t_key] = (best_key, best_sim)
    return matches


def compute_alignment_score(pairs: List[Tuple[float, float, float]]) -> float:
    # Avoid division by zero if all pairs have zero impact
    if not pairs:
        return 0.0

    # Normalize impacts within teacher and student sets
    I_S_sum = sum(I_S for I_S, _, _ in pairs)
    I_T_sum = sum(I_T for _, I_T, _ in pairs)

    normalized_pairs = []
    for I_S, I_T, S in pairs:
        normalized_I_S = I_S / I_S_sum if I_S_sum != 0 else 0
        normalized_I_T = I_T / I_T_sum if I_T_sum != 0 else 0
        normalized_pairs.append((normalized_I_S, normalized_I_T, S))

    return sum(S * (1 - abs(normalized_I_S - normalized_I_T)) for normalized_I_S, normalized_I_T, S in normalized_pairs) / len(normalized_pairs)



teacher_heads = calculate_head_impacts(teacher, teacher_tokenizer, texts, target_token, distractor_token)
student_heads = calculate_head_impacts(student, student_tokenizer, texts, target_token, distractor_token)
teacher_mlp = calculate_mlp_impacts(teacher, teacher_tokenizer, texts, target_token, distractor_token)
student_mlp = calculate_mlp_impacts(student, student_tokenizer, texts, target_token, distractor_token)

influence_teacher = {k: abs(v) for k, v in dict(teacher_heads + teacher_mlp).items()}
influence_student = {k: abs(v) for k, v in dict(student_heads + student_mlp).items()}

teacher_head_vecs = get_head_vectors(teacher, teacher_tokenizer, texts, [x[0] for x in teacher_heads])
student_head_vecs = get_head_vectors(student, student_tokenizer, texts, [x[0] for x in student_heads])

teacher_mlp_vecs = get_mlp_vectors(teacher, teacher_tokenizer, texts, [x[0] for x in teacher_mlp])
student_mlp_vecs = get_mlp_vectors(student, student_tokenizer, texts, [x[0] for x in student_mlp])

attn_matches = compute_best_matches_attn_biased_to_teacher(teacher_head_vecs, student_head_vecs)
mlp_matches = compute_best_matches_mlp_biased_to_teacher(teacher_mlp_vecs, student_mlp_vecs)

matches = {**attn_matches, **mlp_matches}
all_pairs = []
for teacher_comp, (student_comp, sim_score) in matches.items():
    I_T = influence_teacher.get(teacher_comp, 0)
    I_S = influence_student.get(student_comp, 0)
    all_pairs.append((I_S, I_T, sim_score))

alignment_score = compute_alignment_score(all_pairs)


print(f"Alignment score: {alignment_score:.4f}")

Teacher logit difference: 6.1462
Student logit difference: 4.2934


`torch.nn.functional.scaled_dot_product_attention` does not support `output_attentions=True`. Falling back to eager attention. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


Alignment score: 0.9508


In [ ]:
results = compare_teacher_student(
    teacher_heads,
    student_heads,
    [],
    [],
    teacher_params=85_000_000,
    student_params=42_000_000,
)

import pprint, json
pprint.pprint(results, compact=True)
json.dump(results, open("gpt2_compression_robustness.json", "w"), indent=2)


{'compression_C': 0.5058823529411764,
 'robustness_slope_pp_per_C': 11.752541292321148,
 'student': {'heads': {'gini': 0.8387929076365621,
                       'mean': 6.715564741558883,
                       'median': 0.48389473219714874,
                       'single_worst': 90.59627448457454,
                       'worst_k_mean': 62.87244079604553},
             'mean_all': 6.715564741558883,
             'mlp': {'gini': 0.0,
                     'mean': 0.0,
                     'median': 0.0,
                     'single_worst': 0.0,
                     'worst_k_mean': 0.0}},
 'teacher': {'heads': {'gini': 0.824526957988605,
                       'mean': 0.7701614995611262,
                       'median': 0.11878696275313683,
                       'single_worst': 16.37414175952476,
                       'worst_k_mean': 8.603358648175037},
             'mean_all': 0.7701614995611262,
             'mlp': {'gini': 0.0,
                     'mean': 0.0,
                     

# GPT2-small / DistilGPT2 (IOI):

In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, GPT2LMHeadModel
from typing import List, Tuple, Dict
from scipy.linalg import svd
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine_similarity

device = "cuda" if torch.cuda.is_available() else "cpu"

teacher_str = "gpt2"
student_str = "distilgpt2"

teacher_model = GPT2LMHeadModel.from_pretrained(teacher_str).to(device)
teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_str)
teacher_tokenizer.pad_token = teacher_tokenizer.eos_token
teacher_tokenizer.padding_side = "left"

student_model = GPT2LMHeadModel.from_pretrained(student_str).to(device)
student_tokenizer = AutoTokenizer.from_pretrained(student_str)
student_tokenizer.pad_token = student_tokenizer.eos_token
student_tokenizer.padding_side = "left"

teacher_model.eval()
student_model.eval()

import re

raw_data = ['After Mark and Sara went to the office, Mark gave a kiss to Sara', 'Then, Brittany and Bryan had a lot of fun at the house. Brittany gave a snack to Bryan', 'Then, Emily and Kimberly went to the station. Kimberly gave a bone to Emily', 'Then, Christine and Stephanie went to the hospital. Christine gave a snack to Stephanie', 'When Emily and Christine got a necklace at the hospital, Christine decided to give it to Emily', 'Then, Megan and Robert were working at the station. Robert decided to give a basketball to Megan', 'Then, Jacob and Tyler were thinking about going to the house. Tyler wanted to give a necklace to Jacob', 'Then, Erin and Lauren had a lot of fun at the station. Lauren gave a basketball to Erin', 'After Brittany and Kevin went to the hospital, Brittany gave a ring to Kevin', 'Then, James and Nicole had a lot of fun at the house. James gave a snack to Nicole', 'Then, Adam and Bradley had a lot of fun at the station. Adam gave a snack to Bradley', 'When Jessica and Robert got a drink at the school, Robert decided to give it to Jessica', 'When Rachel and Megan got a snack at the office, Rachel decided to give it to Megan', 'Then, Danielle and Nathan had a lot of fun at the office. Danielle gave a necklace to Nathan', 'When Steven and Rebecca got a basketball at the hospital, Steven decided to give it to Rebecca', 'Then, Courtney and Nicole went to the garden. Nicole gave a bone to Courtney', 'After Heather and Alicia went to the hospital, Alicia gave a computer to Heather', 'Then, Nicole and Christina went to the office. Christina gave a bone to Nicole', 'Then, Shannon and Christopher were working at the restaurant. Shannon decided to give a computer to Christopher', 'Then, Joshua and Paul had a lot of fun at the restaurant. Paul gave a snack to Joshua', 'Then, Christopher and Scott went to the station. Scott gave a snack to Christopher', 'When Brittany and Brian got a snack at the hospital, Brian decided to give it to Brittany', 'Then, Daniel and Tiffany went to the garden. Tiffany gave a ring to Daniel', 'Then, Tyler and Sarah had a long argument, and afterwards Tyler said to Sarah', 'Then, Angela and Bradley went to the school. Bradley gave a drink to Angela', 'When Aaron and Kyle got a computer at the house, Kyle decided to give it to Aaron', 'Then, Amanda and Travis were thinking about going to the garden. Amanda wanted to give a basketball to Travis', 'Then, Heather and Kevin had a long argument, and afterwards Kevin said to Heather', 'After Stephanie and Paul went to the restaurant, Paul gave a ring to Stephanie', 'Then, Katie and Charles went to the station. Katie gave a computer to Charles', 'Then, Elizabeth and Timothy had a lot of fun at the hospital. Timothy gave a snack to Elizabeth', 'Then, Jesse and Elizabeth went to the school. Elizabeth gave a drink to Jesse', 'Then, Dustin and Eric went to the restaurant. Eric gave a computer to Dustin', 'Then, Justin and David were working at the restaurant. David decided to give a computer to Justin', 'Then, Tyler and Cody were thinking about going to the station. Tyler wanted to give a kiss to Cody', 'When Katie and Kyle got a computer at the house, Kyle decided to give it to Katie', 'Then, Erin and Dustin had a lot of fun at the garden. Erin gave a ring to Dustin', 'Then, Lauren and Alicia went to the hospital. Lauren gave a ring to Alicia', 'Then, Elizabeth and Alexander went to the store. Alexander gave a computer to Elizabeth', 'Then, Bradley and Jennifer were thinking about going to the restaurant. Bradley wanted to give a snack to Jennifer', 'Then, Aaron and Lindsey had a lot of fun at the restaurant. Aaron gave a drink to Lindsey', 'Then, Sean and Robert had a long argument, and afterwards Robert said to Sean', 'After Jose and Lauren went to the restaurant, Jose gave a necklace to Lauren', 'Then, Erin and Katie went to the house. Erin gave a ring to Katie', 'When Robert and Laura got a kiss at the office, Laura decided to give it to Robert', 'Then, Tiffany and Kelly were thinking about going to the station. Kelly wanted to give a drink to Tiffany', 'Then, Nicole and Samantha had a lot of fun at the station. Samantha gave a snack to Nicole', 'Then, Joshua and Anthony were thinking about going to the restaurant. Anthony wanted to give a computer to Joshua', 'When Lindsay and Sean got a basketball at the store, Sean decided to give it to Lindsay', 'When Rachel and William got a drink at the store, Rachel decided to give it to William', 'Then, Eric and Andrew went to the garden. Andrew gave a bone to Eric', 'When Vanessa and Christina got a drink at the hospital, Christina decided to give it to Vanessa', 'Then, Patrick and Stephen were working at the station. Stephen decided to give a bone to Patrick', 'Then, Gregory and Nicole went to the restaurant. Gregory gave a kiss to Nicole', 'Then, Benjamin and Samuel were working at the hospital. Benjamin decided to give a ring to Samuel', 'Then, Kyle and Courtney were working at the station. Kyle decided to give a bone to Courtney', 'When Sara and Patrick got a drink at the office, Patrick decided to give it to Sara', 'When Lauren and Nicole got a drink at the school, Nicole decided to give it to Lauren', 'After Jeremy and Jennifer went to the station, Jennifer gave a drink to Jeremy', 'Then, Lindsey and Kristen went to the station. Lindsey gave a ring to Kristen', 'Then, Andrea and Jamie had a long argument, and afterwards Andrea said to Jamie', 'Then, Nathan and Rebecca went to the restaurant. Rebecca gave a drink to Nathan', 'Then, Lindsey and Sarah were working at the station. Sarah decided to give a computer to Lindsey', 'After Katie and Travis went to the garden, Katie gave a necklace to Travis', 'Then, Alexander and Sara were working at the office. Sara decided to give a snack to Alexander', 'Then, Dustin and Ashley were working at the garden. Ashley decided to give a basketball to Dustin', 'Then, Jesse and Timothy had a long argument, and afterwards Timothy said to Jesse', 'Then, Bradley and Kenneth went to the garden. Kenneth gave a snack to Bradley', 'Then, Kyle and Katie had a long argument, and afterwards Kyle said to Katie', 'Then, Amanda and Christina were thinking about going to the office. Christina wanted to give a drink to Amanda', 'Then, Nathan and Megan had a long argument, and afterwards Nathan said to Megan', 'Then, Mary and James had a long argument, and afterwards Mary said to James', 'Then, Elizabeth and Justin were thinking about going to the hospital. Elizabeth wanted to give a snack to Justin', 'Then, Christopher and Jose were working at the hospital. Christopher decided to give a bone to Jose', 'Then, Aaron and Danielle went to the hospital. Danielle gave a drink to Aaron', 'When Rebecca and Melissa got a drink at the school, Rebecca decided to give it to Melissa', 'Then, Patrick and Joseph were working at the house. Joseph decided to give a necklace to Patrick', 'Then, Kristen and Jessica were thinking about going to the hospital. Kristen wanted to give a ring to Jessica', 'Then, Cody and Jesse were thinking about going to the station. Cody wanted to give a computer to Jesse', 'Then, Elizabeth and Erica were thinking about going to the school. Erica wanted to give a basketball to Elizabeth', 'Then, Alexander and Melissa were thinking about going to the restaurant. Alexander wanted to give a kiss to Melissa', 'Then, Bryan and Jennifer had a long argument, and afterwards Jennifer said to Bryan', 'Then, Christine and Kimberly were thinking about going to the garden. Christine wanted to give a computer to Kimberly', 'After Benjamin and Cody went to the school, Benjamin gave a bone to Cody', 'Then, Dustin and Kimberly were thinking about going to the hospital. Kimberly wanted to give a bone to Dustin', 'Then, Christina and Patrick were working at the office. Christina decided to give a kiss to Patrick', 'When David and Justin got a drink at the school, Justin decided to give it to David', 'Then, Angela and Kimberly were working at the house. Kimberly decided to give a ring to Angela', 'Then, Ryan and Sara were working at the office. Ryan decided to give a computer to Sara', 'Then, Kenneth and Amy were thinking about going to the garden. Kenneth wanted to give a basketball to Amy', 'When Matthew and Travis got a basketball at the office, Travis decided to give it to Matthew', 'Then, Daniel and Kenneth had a long argument, and afterwards Kenneth said to Daniel', 'Then, Jason and Jennifer went to the hospital. Jennifer gave a snack to Jason', 'Then, John and Adam went to the restaurant. Adam gave a drink to John', 'When Jeffrey and Jose got a bone at the house, Jose decided to give it to Jeffrey', 'Then, John and Lindsey went to the house. John gave a computer to Lindsey', 'Then, Katie and Jeffrey had a lot of fun at the station. Jeffrey gave a snack to Katie', 'After Kimberly and John went to the office, John gave a necklace to Kimberly', 'Then, Michelle and Thomas had a lot of fun at the garden. Thomas gave a snack to Michelle', 'Then, Kristen and Heather went to the store. Heather gave a ring to Kristen']


texts = []
targets = []
distractors = []

pattern = re.compile(
    r"(?i)(?:After|Then|When)?[^.,;]*?\b(\w+)\b and \b(\w+)\b.*?"
    r"\b(\1|\2)\b.*?"
    r"(?:gave|wanted to give|decided to give|said to).*?\bto (\w+)\b"
)

for sentence in raw_data:
    match = pattern.search(sentence)
    if match:
        person1, person2, giver, recipient = match.groups()

        texts.append(sentence.strip().rstrip('.') + ' ')
        targets.append(recipient.lower())
        distractors.append((person1 if recipient.lower() != person1.lower() else person2).lower())

def compute_logit_diff_ioi(model, tokenizer, texts, targets, distractors):
    diffs = []
    for i, text in enumerate(texts):
        inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True).to(device)
        with torch.no_grad():
            logits = model(**inputs).logits

        # Get the logits for the last token in the sequence
        last_token_logits = logits[0, -1, :]

        # Get the token IDs for the target and distractor names
        # Add a space before the name to match how GPT2 tokenizes words
        try:
            target_id = tokenizer.encode(" " + targets[i], add_special_tokens=False)[0]
            distractor_id = tokenizer.encode(" " + distractors[i], add_special_tokens=False)[0]
        except IndexError:
             print(f"Could not tokenize target or distractor for text: {text}")
             continue


        diffs.append((last_token_logits[target_id] - last_token_logits[distractor_id]).item())
    return np.mean(diffs)

teacher_diff = compute_logit_diff_ioi(teacher_model, teacher_tokenizer, texts, targets, distractors)
student_diff = compute_logit_diff_ioi(student_model, student_tokenizer, texts, targets, distractors)

print(f"Teacher logit difference: {teacher_diff:.4f}")
print(f"Student logit difference: {student_diff:.4f}")


def get_logits_gpt2(model, tokenizer, texts):
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs, return_dict=True, output_attentions=True, output_hidden_states=True)
    # Get the logits for the last token
    last_token_logits = outputs.logits[:, -1, :]
    return last_token_logits, inputs, outputs.attentions, outputs.hidden_states

def compute_logit_diff_ioi_logits(logits, tokenizer, targets, distractors):
    diffs = []
    for i in range(logits.shape[0]):
        logits_slice = logits[i, :]
        # Add a space before the name to match how GPT2 tokenizes words
        try:
            target_id = tokenizer.encode(" " + targets[i], add_special_tokens=False)[0]
            distractor_id = tokenizer.encode(" " + distractors[i], add_special_tokens=False)[0]
        except IndexError:
             print(f"Could not tokenize target or distractor for index {i}")
             continue

        diffs.append((logits_slice[target_id] - logits_slice[distractor_id]).item())
    return np.mean(diffs)

def get_ablation_hook(head_idx):
    def ablate_head(module, input, output):
        if isinstance(output, tuple):
            attn_output, *rest = output
            # Ablate the specified head. For GPT2 attention output shape is (batch_size, num_heads, seq_len, seq_len)
            attn_output[:, head_idx] = 0 # Ablate across all queries and keys for this head
            return (attn_output,) + tuple(rest)
        # If output is not a tuple, it's likely the attention output directly (less common)
        output[:, head_idx] = 0
        return output
    return ablate_head

def get_mlp_ablation_hook():
    return lambda module, input, output: torch.zeros_like(output)

def calculate_head_impacts(model, tokenizer, texts, targets, distractors):
    logits, inputs, _, _ = get_logits_gpt2(model, tokenizer, texts)
    baseline_diff = compute_logit_diff_ioi_logits(logits, tokenizer, targets, distractors)

    num_layers = model.config.n_layer
    num_heads = model.config.n_head
    impacts = []

    for layer in range(num_layers):
        # Assuming the attention module is in model.transformer.h[layer].attn
        module = model.transformer.h[layer].attn
        for head in range(num_heads):
            hook = module.register_forward_hook(get_ablation_hook(head))
            logits_ablated, _, _, _ = get_logits_gpt2(model, tokenizer, texts)
            ablated_diff = compute_logit_diff_ioi_logits(logits_ablated, tokenizer, targets, distractors)
            hook.remove()
            # Impact is the change in logit difference relative to the baseline
            impact = (baseline_diff - ablated_diff) / baseline_diff * 100 if baseline_diff != 0 else 0
            impacts.append(((layer, head), impact))
    return impacts

def calculate_mlp_impacts(model, tokenizer, texts, targets, distractors):
    logits, inputs, _, _ = get_logits_gpt2(model, tokenizer, texts)
    baseline_diff = compute_logit_diff_ioi_logits(logits, tokenizer, targets, distractors)
    impacts = []
    for layer in range(model.config.n_layer):
        # Assuming the MLP module is in model.transformer.h[layer].mlp.c_fc
        module = model.transformer.h[layer].mlp.c_fc
        hook = module.register_forward_hook(get_mlp_ablation_hook())
        try:
            logits_ablated, _, _, _ = get_logits_gpt2(model, tokenizer, texts)
            ablated_diff = compute_logit_diff_ioi_logits(logits_ablated, tokenizer, targets, distractors)
            # Impact is the change in logit difference relative to the baseline
            impact = (baseline_diff - ablated_diff) / baseline_diff * 100 if baseline_diff != 0 else 0
            impacts.append((layer, impact))
        finally:
            hook.remove()
    return impacts


def get_head_vectors(model, tokenizer, texts, components: List[Tuple[int, int]]):
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True, return_dict=True)
    attentions = outputs.attentions
    vectors = {}
    for layer, head in components:
        # Get attention weights for the last token across all input tokens
        # Shape: (batch_size, num_heads, seq_len, seq_len)
        # We want the attention from the last token (query) to all previous tokens (keys)
        # So we need attentions[layer][:, head, -1, :]
        # Take the mean across the batch
        vectors[(layer, head)] = attentions[layer][:, head, -1, :].mean(dim=0)
    return vectors


def get_mlp_vectors(model, tokenizer, texts, components: List[int]):
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True, return_dict=True)
    hidden_states = outputs.hidden_states
    vectors = {}
    for layer in components:
        # Get the MLP output for the last token
        # Shape of hidden_states is (layer, batch_size, seq_len, hidden_size)
        # We want hidden_states[layer+1] for the output of layer 'layer'
        # We need the output at the last token position: hidden_states[layer + 1][:, -1, :]
        # Take the mean across the batch
        mlp_output = hidden_states[layer + 1][:, -1, :]
        vectors[layer] = mlp_output.mean(dim=0)
    return vectors

import torch.nn.functional as F

def compute_best_matches_attn_biased_to_teacher(teacher_dict: Dict, student_dict: Dict) -> Dict:
    matches = {}
    for t_key, t_vec in teacher_dict.items():
        best_sim, best_key = -1, None
        for s_key, s_vec in student_dict.items():
            # Adjust for potential sequence length differences
            min_len = min(t_vec.shape[-1], s_vec.shape[-1])
            sim = torch.nn.functional.cosine_similarity(t_vec[-min_len:], s_vec[-min_len:], dim=0).item()
            if sim > best_sim:
                best_sim, best_key = sim, s_key
        matches[t_key] = (best_key, best_sim)
    return matches

def activation_eigenvector_similarity(acts1, acts2, k=3):
    acts1_np = acts1
    acts2_np = acts2

    # Check if there are enough samples for SVD
    min_samples = min(acts1_np.shape[0], acts2_np.shape[0])
    if min_samples < k:
        # print(f"Warning: Not enough samples ({min_samples}) for k={k}. Returning 0 similarity.")
        return 0.0

    cov1 = np.dot(acts1_np.T, acts1_np) / acts1_np.shape[0]
    cov2 = np.dot(acts2_np.T, acts2_np) / acts2_np.shape[0]

    # Handle potential singular matrices
    try:
        U1, S1, V1 = svd(cov1)
        U2, S2, V2 = svd(cov2)
    except np.linalg.LinAlgError:
        # print("Warning: SVD failed. Returning 0 similarity.")
        return 0.0

    # Check if there are enough singular values for k
    if min(S1.shape[0], S2.shape[0]) < k:
         # print(f"Warning: Not enough singular values ({min(S1.shape[0], S2.shape[0])}) for k={k}. Returning 0 similarity.")
         return 0.0


    U1_topk = U1[:, :k]
    U2_topk = U2[:, :k]

    sim_matrix = sklearn_cosine_similarity(U1_topk.T, U2_topk.T)
    return np.mean(sim_matrix)


def compute_best_matches_mlp_biased_to_teacher(teacher_dict: Dict, student_dict: Dict) -> Dict:
    matches = {}
    for t_key, t_vec in teacher_dict.items():
        best_sim, best_key = -1, None
        for s_key, s_vec in student_dict.items():
             # MLP vectors are already meaned over sequence length, so direct comparison is fine
             # activation_eigenvector_similarity expects numpy arrays
            sim = activation_eigenvector_similarity(t_vec.cpu().numpy().reshape(1, -1), s_vec.cpu().numpy().reshape(1, -1))
            if sim > best_sim:
                best_sim, best_key = sim, s_key
        matches[t_key] = (best_key, best_sim)
    return matches


def compute_alignment_score(pairs: List[Tuple[float, float, float]]) -> float:
    # Avoid division by zero if all pairs have zero impact
    if not pairs:
        return 0.0

    # Normalize impacts within teacher and student sets
    I_S_sum = sum(I_S for I_S, _, _ in pairs)
    I_T_sum = sum(I_T for _, I_T, _ in pairs)

    normalized_pairs = []
    for I_S, I_T, S in pairs:
        normalized_I_S = I_S / I_S_sum if I_S_sum != 0 else 0
        normalized_I_T = I_T / I_T_sum if I_T_sum != 0 else 0
        normalized_pairs.append((normalized_I_S, normalized_I_T, S))

    return sum(S * (1 - abs(normalized_I_S - normalized_I_T)) for normalized_I_S, normalized_I_T, S in normalized_pairs) / len(normalized_pairs)


teacher_heads = calculate_head_impacts(teacher_model, teacher_tokenizer, texts, targets, distractors)
student_heads = calculate_head_impacts(student_model, student_tokenizer, texts, targets, distractors)
teacher_mlp = calculate_mlp_impacts(teacher_model, teacher_tokenizer, texts, targets, distractors)
student_mlp = calculate_mlp_impacts(student_model, student_tokenizer, texts, targets, distractors)

influence_teacher = {k: abs(v) for k, v in dict(teacher_heads + [(l, v) for l, v in teacher_mlp]).items()}
influence_student = {k: abs(v) for k, v in dict(student_heads + [(l, v) for l, v in student_mlp]).items()}

student_head_vecs = get_head_vectors(student_model, student_tokenizer, texts, [x[0] for x in student_heads])
teacher_head_vecs = get_head_vectors(teacher_model, teacher_tokenizer, texts, [x[0] for x in teacher_heads])
student_mlp_vecs = get_mlp_vectors(student_model, student_tokenizer, texts, [x[0] for x in student_mlp])
teacher_mlp_vecs = get_mlp_vectors(teacher_model, teacher_tokenizer, texts, [x[0] for x in teacher_mlp])


attn_matches = compute_best_matches_attn_biased_to_teacher(teacher_head_vecs, student_head_vecs)
mlp_matches = compute_best_matches_mlp_biased_to_teacher(teacher_mlp_vecs, student_mlp_vecs)

matches = {**attn_matches, **mlp_matches}
all_pairs = []
for teacher_comp, (student_comp, sim_score) in matches.items():
    I_T = influence_teacher.get(teacher_comp, 0)
    I_S = influence_student.get(student_comp, 0)
    all_pairs.append((I_S, I_T, sim_score))

alignment_score = compute_alignment_score(all_pairs)

print(f"Alignment score: {alignment_score:.4f}")

`torch.nn.functional.scaled_dot_product_attention` does not support `output_attentions=True`. Falling back to eager attention. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


Teacher logit difference: 0.0903
Student logit difference: -0.0771
Alignment score: 0.8778


In [ ]:
results = compare_teacher_student(
    teacher_heads,
    student_heads,
    [],
    [],
    teacher_params=85_000_000,
    student_params=42_000_000,
)

import pprint, json
pprint.pprint(results, compact=True)
json.dump(results, open("gpt2_compression_robustness.json", "w"), indent=2)


{'compression_C': 0.5058823529411764,
 'robustness_slope_pp_per_C': 15.337674019132377,
 'student': {'heads': {'gini': 0.4362425967481705,
                       'mean': 12.924138309243126,
                       'median': 0.0,
                       'single_worst': 212.48037954728161,
                       'worst_k_mean': 147.56913394449157},
             'mean_all': 12.924138309243126,
             'mlp': {'gini': 0.0,
                     'mean': 0.0,
                     'median': 0.0,
                     'single_worst': 0.0,
                     'worst_k_mean': 0.0}},
 'teacher': {'heads': {'gini': 0.24103300155931784,
                       'mean': 5.165079687799689,
                       'median': 0.023291518703580184,
                       'single_worst': 105.93971729521819,
                       'worst_k_mean': 74.67895139120994},
             'mean_all': 5.165079687799689,
             'mlp': {'gini': 0.0,
                     'mean': 0.0,
                     'median': 